# Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cloning the Repo

In [ ]:
import os
from getpass import getpass

In [ ]:
%cd /content/
GITHUB_TOKEN= getpass('Enter your GitHub Personal Access Token:')
repo_url = f"https://{GITHUB_TOKEN}@github.com/Speech-NeuroAI-GP/NeuroAI-Toolkit.git"

In [ ]:
!git clone {repo_url}

# 6. Change into the cloned repository directory
%cd NeuroAI-Toolkit

In [ ]:
del GITHUB_TOKEN

# Downloading Workbench

In [ ]:
%cd /content/NeuroAI-Toolkit

# Make the installation script executable
!chmod +x install_workbench.sh

# Run the Workbench installation script
!./install_workbench.sh
os.environ['PATH'] += ":/usr/local/workbench/bin_linux64"

# Installing the Package

In [ ]:
%cd /content/NeuroAI-Toolkit

# Install the package and its dependencies
!pip install -e .

# Cleaning the Data Using Our Cleaning Module

In [ ]:
from neursen_reveal.fmri_cleaning import run_cleaning_pipeline
run_cleaning_pipeline(
    bids_fmriprep_root_dir= "/content/drive/MyDrive/Moth_Radio_Hour/Moth_Radio_hour_Raw",
    output_base_dir= "/content/drive/MyDrive/Moth_Radio_Hour/cleaned",
    left_surface_template= "/content/drive/MyDrive/Moth_Radio_Hour/surface/tpl-fsLR_hemi-L_den-32k_sphere.surf.gii",
    right_surface_template= "/content/drive/MyDrive/Moth_Radio_Hour/surface/tpl-fsLR_hemi-L_den-32k_sphere.surf.gii",
    selected_confounds= ['trans_x', 'trans_y', 'trans_z',
                   'rot_x', 'rot_y', 'rot_z', ],
    smoothing_kernel_size = 2
)

# You can also specify certain subjects and/or sessions and/or tasks from the data folder to clean
run_cleaning_pipeline(
    bids_fmriprep_root_dir= "/content/drive/MyDrive/Moth_Radio_Hour/Moth_Radio_hour_Raw",
    output_base_dir= "/content/drive/MyDrive/Moth_Radio_Hour/cleaned",
    left_surface_template= "/content/drive/MyDrive/Moth_Radio_Hour/surface/tpl-fsLR_hemi-L_den-32k_sphere.surf.gii",
    right_surface_template= "/content/drive/MyDrive/Moth_Radio_Hour/surface/tpl-fsLR_hemi-L_den-32k_sphere.surf.gii",
    selected_confounds= ['trans_x', 'trans_y', 'trans_z',
                   'rot_x', 'rot_y', 'rot_z', ],
    smoothing_kernel_size = 2,
    target_subject_ids= 'sub-UTS01',
    target_session_ids= 'ses-5',
    target_task_ids= 'task-adollshouse',
)

# Extract Stimuli Features of the AI Model of Interset Using Our Feature Extraction Module

In [ ]:
from neursen_reveal.feature_extraction import extract_features
extract_features(bold_data_root_dir= "/content/drive/MyDrive/Moth_Radio_Hour/cleaned",
                 stimuli_base_dir= "/content/drive/MyDrive/Moth_Radio_Hour/padded_stimuli_stories",
                 output_file_path= "/content/drive/MyDrive/Moth_Radio_Hour/features/ecapa_tdnn.h5",
                 model_id= "ecapa-tdnn",
                 tr= 2)

# Apply Linear Mapping Between AI Model Features and Brain Activations Using Our Voxelwise Modeling Module

In [ ]:
from neursen_reveal.voxelwise_modeling import generate_scores_for_specific_model_across_subjects, render_scores

generate_scores_for_specific_model_across_subjects(
    Bold_dataset_dir= "/content/drive/MyDrive/Moth_Radio_Hour/cleaned",
    features_file_path= "/content/drive/MyDrive/Moth_Radio_Hour/features/ecapa_tdnn.h5",
    saving_dir= "/content/drive/MyDrive/Moth_Radio_Hour/scores",
    bold_test= ['wheretheressmoke'],
)

# Perform Statistical Significance Testing on the Results Across Subjects Using Our Stats Module

In [ ]:
from neursen_reveal.stats import get_significant_model_scores
significant_scores_map = get_significant_model_scores(
                results_dir="/content/drive/MyDrive/Moth_Radio_Hour/scores",
                alpha=0.05 # Optional, Defaults to 0.05
            )
significant_scores_path = '/content/drive/MyDrive/Moth_Radio_Hour/significant_scores.npy'
np.save(significant_scores_path, significant_scores_map)

# Plot Significant Brain Scores on Brain Surface Using Our Rendering Feature From the Voxelwise Modeling Module

In [ ]:
render_scores(scores= significant_scores_map, threshold= "90%", title = "Significant Averaged Rendered R-squared Scores")

# Parcellate Brain Scores to 17 Brain Functional Networks Using Our Parcellation Module

In [ ]:
from neursen_reveal.parcellation import plot_parcellation_of_peak_intensity_regions
plot_parcellation_of_peak_intensity_regions(
                data=significant_scores_map,
                threshold= "90%"
            )

# Or Directly Use Our Full Unified Voxelwise Modeling Pipeline

In [ ]:
from neursen_reveal.full_pipeline import run_full_voxelwise_pipeline
run_full_voxelwise_pipeline(
    bids_data_root_dir= "/content/drive/MyDrive/Moth_Radio_Hour/Moth_Radio_hour_cleaned",
    output_base_dir= '/content/drive/MyDrive/Moth_Radio_Hour/output',
    voxelwise_model_bold_test_runs=['wheretheressmoke'],
    stimuli_base_dir= "/content/drive/MyDrive/Moth_Radio_Hour/padded_stimuli_stories",
    ai_model_id= 'ecapa-tdnn',
    ai_layer_name='final',
    tr = 2.0,
    voxelwise_model_delayers = [1, 2, 3, 4],
    voxelwise_model_n_runs_out= 1,
    perform_cleaning= True,
    selected_confounds= ['trans_x', 'trans_y', 'trans_z',
                   'rot_x', 'rot_y', 'rot_z', ],
    smoothing_kernel_size= 2,
    left_surface_template= "/content/drive/MyDrive/Moth_Radio_Hour/surface/tpl-fsLR_hemi-L_den-32k_sphere.surf.gii",
    right_surface_template= "/content/drive/MyDrive/Moth_Radio_Hour/surface/tpl-fsLR_hemi-L_den-32k_sphere.surf.gii",
    statistical_significance_testing= True,
    significance_alpha= 0.05,
    plot_threshold= "90%",
)

# Compute RSA Using Our RSA Module

In [ ]:
from neursen_reveal.rsa import run_rsa_pipeline
features_rdm, bold_rdm, rsa_score= run_rsa_pipeline(
    bold_data_root_dir= "/content/drive/MyDrive/Moth_Radio_Hour/Moth_Radio_hour_cleaned",
    functional_run_name= 'inamoment',
    precomputed_features_h5_path= '/content/drive/My Drive/Moth_Radio_Hour/features/ecapa_features/ecapa_features.h5'
    network_of_interest= 'auditory',
    h_delay_trs= 2,
    perform_statistical_significance_testing= True,
    rsa_metric= 'cosine',
    plot_rdms= True
)

In [ ]:
features_rdm, bold_rdm, rsa_score= run_rsa_pipeline(
    bold_data_root_dir= "/content/drive/MyDrive/Moth_Radio_Hour/Moth_Radio_hour_cleaned",
    functional_run_name= 'inamoment'
    network_of_interest= 'auditory',
    h_delay_trs= 2,
    perform_statistical_significance_testing= True,
    extract_features_on_the_fly= True,
    stimuli_base_dir= "/content/drive/MyDrive/Moth_Radio_Hour/padded_stimuli_stories",
    ai_model_id= 'ecapa-tdnn',
    ai_layer_name='final',
    rsa_metric= 'cosine',
    plot_rdms= True
)